# MapBiomas Argentina — Fuego Colección 1
## Paso 04 · segmentación SNIC de área quemada — exportación distribuida

**Cómo correrlo:** poné el rango de años en la **Celda 3** (`year_start` y `year_end`, ambos inclusive — p. ej. `year_start = 2005`, `year_end = 2010`) y ejecutá las celdas en orden **1 → 2 → 3 → 4**. Autenticate con tu cuenta de Google cuando lo pida (Celda 2). Cada año es una tarea de país entero que se envía a los servidores de GEE con `overwrite=True` (reemplaza el asset si ya existe); una vez enviadas podés cerrar la pestaña. Los años son **fire-years** nombrados por su año de inicio (FY = 1 may Y1 → 30 abr Y2).

> Corre en el proyecto de cómputo `mapbiomas-argentina`. Repartimos los años entre 4 cuentas; coordinamos por WhatsApp qué rango toma cada uno.

In [ ]:
# Celda 1 — Configuración. Ejecutar una vez por sesión (también al volver si Colab caduca).
!pip install -q -U earthengine-api
!git clone --depth 1 -b main https://github.com/barberaivan/mapbiomas-argentina-fire.git 2>/dev/null || (cd mapbiomas-argentina-fire && git pull -q)

import importlib.util
import sys
sys.path.insert(0, '/content/mapbiomas-argentina-fire/collection-01')

# El código del paso 04 vive en workflow/04-snic.py. Como el nombre empieza con un dígito
# y tiene guion, Python no lo puede importar por nombre; se carga por RUTA con importlib.
# Queda en `S`, así que las celdas de abajo usan S.process_fire_year(...).
_ruta = '/content/mapbiomas-argentina-fire/collection-01/workflow/04-snic.py'
_spec = importlib.util.spec_from_file_location('step04_snic', _ruta)
S = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(S)
print('repositorio clonado, funciones importadas — listo')

In [ ]:
# Celda 2 — Autenticación. Iniciá sesión con tu cuenta de Google (la habilitada para el proyecto).
# Al volver después de que Colab caduque hay que autenticarse de nuevo.
# NOTA: Google te va a pedir permisos amplios, incluido Drive. Es parte del login estándar de
# Earth Engine; aceptá. Este notebook solo escribe en assets de GEE, NO toca tu Drive.
GEE_PROJECT = 'mapbiomas-argentina'

import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print('inicializado — proyecto de cómputo:', GEE_PROJECT)

In [ ]:
# Celda 3 — ✏️ Poné acá el rango de años a exportar (ambos inclusive).
year_start = 2005
year_end   = 2010

In [ ]:
# Celda 4 — Enviar la exportación de cada fire-year del rango (con overwrite=True).
from utils import constants as C

region = ee.FeatureCollection(C.ARG_BUFFER_FC).geometry()
proj = ee.Image(ee.ImageCollection(C.BP_TS_METRICS_COL).first()).projection()
crs = proj.crs().getInfo()
transform = proj.getInfo()['transform']

for fy in range(year_start, year_end + 1):
    S.process_fire_year(fy, region, crs, transform,
                        launch=True, name_prefix='snic_', overwrite=True)

print('listo — tareas enviadas para', year_start, 'a', year_end,
      '— corren en los servidores de GEE; podés cerrar esta pestaña.')